<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Fraud-Detection-with-Federated-Learning/blob/main/credit_card.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================
# Credit Card Fraud Detection using DNNs (No FL, FedAvg, FedBN)
# ================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load and preprocess dataset
# -----------------------------
import os

os.environ['KAGGLE_USERNAME'] = "devindithathsara "
os.environ['KAGGLE_KEY'] = "29219555"

print("⏳ Downloading dataset from Kaggle...")
!pip install -q kaggle
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip -d dataset/

import pandas as pd
from sklearn.preprocessing import StandardScaler

print("⏳ Loading and preprocessing dataset...")
df = pd.read_csv('dataset/creditcard.csv')  # replace with your path
print("Dataset shape:", df.shape)

# Separate features and label
X = df.drop(columns=["Class"]).values
y = df["Class"].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

INPUT_DIM = X.shape[1]

# -----------------------------
# 2. Split data
# -----------------------------
NUM_CLIENTS = 12
alpha = 0.5
TEST_SIZE = 0.2

def dirichlet_split(X, y, num_clients, alpha):
    data_per_client = [[] for _ in range(num_clients)]
    labels = np.unique(y)
    for label in labels:
        idx = np.where(y == label)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        split_idx = np.split(idx, proportions)
        for i in range(num_clients):
            data_per_client[i].extend(split_idx[i])
    clients_data = {}
    for i in range(num_clients):
        client_idx = data_per_client[i]
        clients_data[i] = {"X": X[client_idx], "y": y[client_idx]}
    return clients_data

# Train-test split normally
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y)

# Non-IID client splits (training only)
raw_clients_train = dirichlet_split(X_train_full, y_train_full, NUM_CLIENTS, alpha)

# -----------------------------
# 3. Prepare client datasets
# -----------------------------
clients_train_data = {}
clients_test_data = {}

for i in range(NUM_CLIENTS):
    X_c = raw_clients_train[i]["X"]
    y_c = raw_clients_train[i]["y"]
    clients_train_data[i] = {"X": X_c.astype('float32'), "y": y_c.astype('float32')}
    clients_test_data[i] = {"X": X_test_full.astype('float32'), "y": y_test_full.astype('float32')}

# -----------------------------
# 4. DNN Model
# -----------------------------
def create_model(input_dim=INPUT_DIM):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# -----------------------------
# 5. Local Training (No FL)
# -----------------------------
LOCAL_EPOCHS = 5
BATCH_SIZE = 2048

print("\n--- Training DNNs Locally (No FL) ---")
client_models_noFL = []

for cid in range(NUM_CLIENTS):
    print(f"Training Client {cid+1} locally (epochs={LOCAL_EPOCHS}, batch_size={BATCH_SIZE})...")
    model = create_model()
    model.fit(
        clients_train_data[cid]["X"], clients_train_data[cid]["y"],
        epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
    )
    client_models_noFL.append(model)

# Evaluate local models
print("\n--- Local Model Accuracy ---")
local_accs = []
for cid, model in enumerate(client_models_noFL):
    loss, acc = model.evaluate(clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0)
    local_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")
print(f"Average Local Accuracy: {np.mean(local_accs):.4f}")

# -----------------------------
# 6. FedAvg Training
# -----------------------------
NUM_ROUNDS = 20

def fedavg_aggregate(client_weights):
    new_weights = []
    for weights in zip(*client_weights):
        new_weights.append(np.mean(weights, axis=0))
    return new_weights

print("\n--- Starting FedAvg Training ---")
global_model_fedavg = create_model()

for r in range(NUM_ROUNDS):
    print(f"\n--- Global Round {r+1}/{NUM_ROUNDS} ---")
    client_weights = []

    for cid in range(NUM_CLIENTS):
        local_model = create_model()
        local_model.set_weights(global_model_fedavg.get_weights())
        print(f" Client {cid+1} training (epochs={LOCAL_EPOCHS}, batch_size={BATCH_SIZE})...")
        local_model.fit(
            clients_train_data[cid]["X"], clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
        )
        client_weights.append(local_model.get_weights())

    global_model_fedavg.set_weights(fedavg_aggregate(client_weights))
    print(f"Round {r+1} aggregation complete.")

# Evaluate FedAvg model
print("\n--- FedAvg Global Model Accuracy ---")
fedavg_accs = []
for cid in range(NUM_CLIENTS):
    loss, acc = global_model_fedavg.evaluate(clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0)
    fedavg_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")
print(f"Average FedAvg Accuracy: {np.mean(fedavg_accs):.4f}")

# -----------------------------
# 7. FedBN Training
# -----------------------------
print("\n--- Starting FedBN Training ---")
global_model_fedbn = create_model()
client_models_fedbn = [create_model() for _ in range(NUM_CLIENTS)]
for m in client_models_fedbn:
    m.set_weights(global_model_fedbn.get_weights())

def sync_non_bn(global_model, local_model):
    for g_layer, l_layer in zip(global_model.layers, local_model.layers):
        if 'batch_normalization' not in g_layer.name.lower():
            l_layer.set_weights(g_layer.get_weights())

def fedbn_aggregate(client_models, global_model):
    new_weights = []
    for i, layer in enumerate(global_model.layers):
        if 'batch_normalization' not in layer.name.lower():
            weights_all_clients = [m.layers[i].get_weights() for m in client_models]
            avg_weights = [np.mean(w, axis=0) for w in zip(*weights_all_clients)]
            new_weights.extend(avg_weights)
        else:
            new_weights.extend(layer.get_weights())
    return new_weights

for r in range(NUM_ROUNDS):
    print(f"\n--- Global Round {r+1}/{NUM_ROUNDS} ---")
    for cid in range(NUM_CLIENTS):
        print(f" Client {cid+1} training (epochs={LOCAL_EPOCHS}, batch_size={BATCH_SIZE})...")
        local_model = client_models_fedbn[cid]
        sync_non_bn(global_model_fedbn, local_model)
        local_model.fit(
            clients_train_data[cid]["X"], clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
        )
    global_model_fedbn.set_weights(fedbn_aggregate(client_models_fedbn, global_model_fedbn))

# Evaluate FedBN models
print("\n--- FedBN Client Models Accuracy ---")
fedbn_accs = []
for cid in range(NUM_CLIENTS):
    loss, acc = client_models_fedbn[cid].evaluate(clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0)
    fedbn_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")
print(f"Average FedBN Accuracy: {np.mean(fedbn_accs):.4f}")

Streaming output truncated to the last 5000 lines.
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.0769 - loss: 1.7586
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.0769 - loss: 1.7105
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.0577 - loss: 1.6708
 Client 2 training (epochs=5, batch_size=2048)...
Epoch 1/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9994 - loss: 0.1745
Epoch 2/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9999 - loss: 0.1010
Epoch 3/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 1.0000 - loss: 0.0614
Epoch 4/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 1.0000 - loss: 0.0399
Epoch 5/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0276
 Client 3 training (epochs=5, batch_size=2048)...
Epoch 1/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9980 - loss: 0.1957
Epoch 2/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9985 - loss: 0.1723
Epoch 3/5
8/8 ━━━━━

#not training like client 1 epochs 5 client 2 epochs 5 instead of that all the clients for 5 epochs - 20 global rounds

In [1]:
# ================================================
# Credit Card Fraud Detection using DNNs (No FL, FedAvg, FedBN)
# ================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load and preprocess dataset
# -----------------------------
import os

os.environ['KAGGLE_USERNAME'] = "devindithathsara "
os.environ['KAGGLE_KEY'] = "29219555"

print("⏳ Downloading dataset from Kaggle...")
!pip install -q kaggle
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip -d dataset/

import pandas as pd
from sklearn.preprocessing import StandardScaler

print("⏳ Loading and preprocessing dataset...")
df = pd.read_csv('dataset/creditcard.csv')  # replace with your path
print("Dataset shape:", df.shape)

# Separate features and label
X = df.drop(columns=["Class"]).values
y = df["Class"].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

INPUT_DIM = X.shape[1]

# -----------------------------
# 2. Split data
# -----------------------------
NUM_CLIENTS = 12
alpha = 0.5
TEST_SIZE = 0.2

def dirichlet_split(X, y, num_clients, alpha):
    data_per_client = [[] for _ in range(num_clients)]
    labels = np.unique(y)
    for label in labels:
        idx = np.where(y == label)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        split_idx = np.split(idx, proportions)
        for i in range(num_clients):
            data_per_client[i].extend(split_idx[i])
    clients_data = {}
    for i in range(num_clients):
        client_idx = data_per_client[i]
        clients_data[i] = {"X": X[client_idx], "y": y[client_idx]}
    return clients_data

# Train-test split normally
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y)

# Non-IID client splits (training only)
raw_clients_train = dirichlet_split(X_train_full, y_train_full, NUM_CLIENTS, alpha)

# -----------------------------
# 3. Prepare client datasets
# -----------------------------
clients_train_data = {}
clients_test_data = {}

for i in range(NUM_CLIENTS):
    X_c = raw_clients_train[i]["X"]
    y_c = raw_clients_train[i]["y"]
    clients_train_data[i] = {"X": X_c.astype('float32'), "y": y_c.astype('float32')}
    clients_test_data[i] = {"X": X_test_full.astype('float32'), "y": y_test_full.astype('float32')}

# -----------------------------
# 4. DNN Model
# -----------------------------
def create_model(input_dim=INPUT_DIM):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


⏳ Downloading dataset from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:03<00:00, 21.3MB/s]

Archive:  creditcardfraud.zip
  inflating: dataset/creditcard.csv  
⏳ Loading and preprocessing dataset...
Dataset shape: (284807, 31)


In [2]:
print("\n==============================")
print("TRAINING WITHOUT FEDERATED LEARNING")
print("==============================")

LOCAL_EPOCHS = 5
BATCH_SIZE = 2048

client_models_noFL = []

for cid in range(NUM_CLIENTS):
    print(f"\nClient {cid+1} Local Training")
    model = create_model()
    model.fit(
        clients_train_data[cid]["X"], clients_train_data[cid]["y"],
        epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
    )
    client_models_noFL.append(model)

print("\n--- Local Model Accuracy ---")
local_accs = []
for cid, model in enumerate(client_models_noFL):
    loss, acc = model.evaluate(clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0)
    local_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")

print(f"\n✅ Average Local Accuracy: {np.mean(local_accs):.4f}")


TRAINING WITHOUT FEDERATED LEARNING

Client 1 Local Training
Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3684 - loss: 1.1523
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.3860 - loss: 1.0266
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4737 - loss: 0.9184
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5614 - loss: 0.8229
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5789 - loss: 0.7368

Client 2 Local Training
Epoch 1/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5438 - loss: 0.7317
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6179 - loss: 0.6636
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7280 - loss: 0.6173
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8354 - loss: 0.5753
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9117 - loss: 0.5321

Client 3 Local Training
Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 

In [ ]:
print("\n==============================")
print("FEDERATED LEARNING WITH FedAvg")
print("==============================")

NUM_ROUNDS = 20
round_accs_fedavg = []

global_model_fedavg = create_model()

def fedavg_aggregate(client_weights):
    new_weights = []
    for weights in zip(*client_weights):
        new_weights.append(np.mean(weights, axis=0))
    return new_weights

for r in range(NUM_ROUNDS):
    print(f"\n Global Round {r+1}/{NUM_ROUNDS}")

    client_weights = []

    # Train all clients silently
    for cid in range(NUM_CLIENTS):
        local_model = create_model()
        local_model.set_weights(global_model_fedavg.get_weights())

        local_model.fit(
            clients_train_data[cid]["X"], clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0
        )

        client_weights.append(local_model.get_weights())

    # Aggregate
    global_model_fedavg.set_weights(fedavg_aggregate(client_weights))

    # Evaluate global model
    accs = []
    for cid in range(NUM_CLIENTS):
        loss, acc = global_model_fedavg.evaluate(
            clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0
        )
        accs.append(acc)

    avg_acc = np.mean(accs)
    round_accs_fedavg.append(avg_acc)
    print(f" Round {r+1} Average Accuracy: {avg_acc:.4f}")

print(f"\n Final FedAvg Accuracy: {round_accs_fedavg[-1]:.4f}")


FEDERATED LEARNING WITH FedAvg

 Global Round 1/20
 Round 1 Average Accuracy: 0.9709

 Global Round 2/20
 Round 2 Average Accuracy: 0.9950

 Global Round 3/20
 Round 3 Average Accuracy: 0.9985

 Global Round 4/20
 Round 4 Average Accuracy: 0.9992

 Global Round 5/20
 Round 5 Average Accuracy: 0.9993

 Global Round 6/20
 Round 6 Average Accuracy: 0.9992

 Global Round 7/20
 Round 7 Average Accuracy: 0.9992

 Global Round 8/20


In [ ]:
print("\n==============================")
print("FEDERATED LEARNING WITH FedBN")
print("==============================")

round_accs_fedbn = []

global_model_fedbn = create_model()
client_models_fedbn = [create_model() for _ in range(NUM_CLIENTS)]

for m in client_models_fedbn:
    m.set_weights(global_model_fedbn.get_weights())

def sync_non_bn(global_model, local_model):
    for g_layer, l_layer in zip(global_model.layers, local_model.layers):
        if 'batch_normalization' not in g_layer.name.lower():
            l_layer.set_weights(g_layer.get_weights())

def fedbn_aggregate(client_models, global_model):
    new_weights = []
    for i, layer in enumerate(global_model.layers):
        if 'batch_normalization' not in layer.name.lower():
            weights_all_clients = [m.layers[i].get_weights() for m in client_models]
            avg_weights = [np.mean(w, axis=0) for w in zip(*weights_all_clients)]
            new_weights.extend(avg_weights)
        else:
            new_weights.extend(layer.get_weights())
    return new_weights

for r in range(NUM_ROUNDS):
    print(f"\n🌍 Global Round {r+1}/{NUM_ROUNDS}")

    # Train all clients silently
    for cid in range(NUM_CLIENTS):
        local_model = client_models_fedbn[cid]
        sync_non_bn(global_model_fedbn, local_model)

        local_model.fit(
            clients_train_data[cid]["X"], clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0
        )

    # Aggregate non-BN layers
    global_model_fedbn.set_weights(
        fedbn_aggregate(client_models_fedbn, global_model_fedbn)
    )

    # Evaluate client models
    accs = []
    for cid in range(NUM_CLIENTS):
        loss, acc = client_models_fedbn[cid].evaluate(
            clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0
        )
        accs.append(acc)

    avg_acc = np.mean(accs)
    round_accs_fedbn.append(avg_acc)
    print(f"✅ Round {r+1} Average Accuracy: {avg_acc:.4f}")

print(f"\n🏁 Final FedBN Accuracy: {round_accs_fedbn[-1]:.4f}")